# Contract Clause Extractor

**NLP IE — Identify payment, termination, liability clauses in contracts**

## Project Overview

Build a contract clause classifier using **keyword matching** to identify
**payment**, **termination**, **liability**, **confidentiality**, and
**governing law** clauses from legal text.

## Learning Objectives

1. Classify text by legal clause type using keywords.
2. Handle legal terminology.
3. Evaluate clause detection accuracy.

## Problem Statement

Given contract paragraphs, classify each by clause type.

## Why This Project Matters

- Contract review is time-consuming.
- Missing a key clause has legal consequences.
- Legal AI is a growing market.

## Dataset Overview

12 synthetic contract paragraphs with labeled clause types.

## Dataset Source and License Notes

Synthetic legal text.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas", "numpy", "matplotlib", "seaborn"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
CLAUSE_KEYWORDS = {
    "PAYMENT": ["payment","invoice","net 30","fee","compensation","payable","installment","billing"],
    "TERMINATION": ["termination","terminate","cancellation","cancel","expiration","notice period"],
    "LIABILITY": ["liability","indemnify","indemnification","hold harmless","damages","negligence"],
    "CONFIDENTIALITY": ["confidential","non-disclosure","proprietary","trade secret","nda"],
    "GOVERNING_LAW": ["governing law","jurisdiction","venue","governed by","laws of the state"],
}
print(f"Clause types: {list(CLAUSE_KEYWORDS.keys())}")


## Dataset Download or Loading

In [ ]:
PARAGRAPHS = [
    {"text": "The Client shall pay a fee of $50,000 payable in monthly installments. Payment due Net 30.", "true_clause": "PAYMENT"},
    {"text": "Either party may terminate this agreement with 30 days written notice.", "true_clause": "TERMINATION"},
    {"text": "The Contractor shall indemnify and hold harmless the Client against any claims from negligence.", "true_clause": "LIABILITY"},
    {"text": "All information exchanged shall be treated as confidential. Neither party shall disclose proprietary information.", "true_clause": "CONFIDENTIALITY"},
    {"text": "This agreement shall be governed by the laws of the State of California.", "true_clause": "GOVERNING_LAW"},
    {"text": "Total compensation shall be $120,000 billed quarterly. Late payments incur 1.5% monthly fee.", "true_clause": "PAYMENT"},
    {"text": "This contract shall expire on Dec 31, 2026. Early cancellation requires 60 days notice and a termination fee.", "true_clause": "TERMINATION"},
    {"text": "Neither party shall be liable for consequential damages. Total liability shall not exceed fees paid.", "true_clause": "LIABILITY"},
    {"text": "The receiving party agrees to maintain strict confidentiality of all trade secrets for 5 years.", "true_clause": "CONFIDENTIALITY"},
    {"text": "This agreement is governed by the laws of New York. Exclusive jurisdiction in New York County.", "true_clause": "GOVERNING_LAW"},
    {"text": "The parties agree to cooperate in good faith.", "true_clause": "OTHER"},
    {"text": "All work delivered shall be original and free of third-party IP claims.", "true_clause": "OTHER"},
]
df = pd.DataFrame(PARAGRAPHS)
print(f"Loaded {len(df)} paragraphs")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)} paragraphs")


## Exploratory Data Analysis

In [ ]:
print(df["true_clause"].value_counts())


## Target Analysis

Multi-class paragraph classification.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

Keyword matching per clause type.

## Feature Engineering — Clause Classifier

In [ ]:
def classify_clause(text):
    text_lower = text.lower()
    scores = {ct: sum(1 for kw in kws if kw in text_lower) for ct, kws in CLAUSE_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "OTHER"
print("classify_clause() ready.")


## Baseline Model — Classify All Paragraphs

In [ ]:
df["predicted"] = df["text"].apply(classify_clause)
accuracy = (df["true_clause"] == df["predicted"]).mean()
print(f"Overall accuracy: {accuracy:.1%}")
for _, row in df.iterrows():
    s = "✓" if row["true_clause"] == row["predicted"] else "✗"
    print(f"  {s} True: {row["true_clause"]:18s} Pred: {row["predicted"]:18s}")


## LazyPredict Benchmark

> **Not applicable.** This is an NLP IE task. LazyPredict is not used.

## FLAML AutoML Run

> **Not applicable.** FLAML is not used for NLP IE tasks.

## Additional Best-Library Workflow

In [ ]:
from sklearn.metrics import classification_report
labels = sorted(df["true_clause"].unique())
print(classification_report(df["true_clause"], df["predicted"], labels=labels, zero_division=0))


## Top 3 Model Selection

Single keyword classifier.

## Final Training and Evaluation of Top 3

Keyword classifier is the final system.

## Error Analysis

In [ ]:
errors = df[df["true_clause"] != df["predicted"]]
print(f"Errors: {len(errors)}/{len(df)}")


## Interpretation and Business Insight

1. Keyword matching works well for standard legal boilerplate.
2. Payment and liability clauses have distinctive vocabulary.
3. Real contracts require ML-based approaches.

## Limitations

- Keyword lists not exhaustive.
- No implicit clause detection.
- English-only.

## How to Improve This Project

1. Use **transformer classification** (ModernBERT).
2. Add **clause boundary detection**.
3. Use **GLiNER** for zero-shot extraction.

## Production Considerations

- Integrate with contract management systems.
- Human review for flagged clauses.
- Multiple languages and jurisdictions.

## Common Mistakes

1. Assuming consistent formatting.
2. Missing negated clauses.
3. Not handling cross-references.

## Mini Challenge / Exercises

1. Add FORCE_MAJEURE clause type.
2. Test on a real open-source contract.
3. Add confidence scores.

## Final Summary / Key Takeaways

1. **Keyword-based clause detection** is effective for standard text.
2. **Real contracts require ML models.**
3. **Always validate with legal experts.**